In [0]:
%sql
USE CATALOG DataWarehouse;

In [0]:
%sql
SELECT * FROM INFORMATION_SCHEMA.COLUMNS;

table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,full_data_type,data_type,character_maximum_length,character_octet_length,numeric_precision,numeric_precision_radix,numeric_scale,datetime_precision,interval_type,interval_precision,maximum_cardinality,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_system_time_period_start,is_system_time_period_end,system_time_period_timestamp_generation,is_updatable,partition_index,comment
datawarehouse,bronze,crm_cust_info,cst_id,0,null,YES,int,INT,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_key,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_firstname,2,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_lastname,3,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_marital_status,4,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_gndr,5,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_cust_info,cst_create_date,6,null,YES,date,DATE,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_prd_info,prd_id,0,null,YES,int,INT,null,null,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_prd_info,prd_key,1,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null
datawarehouse,bronze,crm_prd_info,prd_nm,2,null,YES,string,STRING,0,0,null,null,null,null,null,null,null,NO,null,null,null,null,null,null,NO,null,NO,NO,null,YES,null,null


In [0]:
%sql
SELECT 'Total Sales' as measure_name, SUM(sales_amount) as measure_value
FROM gold.fact_sales
UNION ALL
SELECT 'Total Quantyty' as measure_name, SUM(quantity) as measure_value
FROM gold.fact_sales
UNION ALL
SELECT 'Avg Price' as measure_name, AVG(price) as measure_value
FROM gold.fact_sales
UNION ALL
SELECT 'Total Num Orders' as measure_name, COUNT(DISTINCT(order_number)) as measure_value
FROM gold.fact_sales
UNION ALL
SELECT 'Total Num Products' as measure_name, COUNT(DISTINCT(product_key)) as measure_value
FROM gold.fact_sales
UNION ALL
SELECT 'Total Num Customers' as measure_name, COUNT(cusotmer_key) as measure_value
FROM gold.fact_sales;

measure_name,measure_value
Total Sales,2.935625E7
Total Quantyty,60423.0
Avg Price,486.0377827080367
Total Num Orders,27659.0
Total Num Products,130.0
Total Num Customers,60398.0


In [0]:
%sql
SELECT country, COUNT(customer_number) as total_customer FROM datawarehouse.gold.dim_customers
GROUP BY country
ORDER BY total_customer DESC;

country,total_customer
United States,7482
Australia,3591
United Kingdom,1913
France,1810
Germany,1780
Canada,1571
n/a,337


In [0]:
%sql
SELECT gender, COUNT(customer_number) as total_customer FROM datawarehouse.gold.dim_customers
GROUP BY gender
ORDER BY total_customer DESC;

gender,total_customer
Male,9341
Female,9128
n/a,15


In [0]:
%sql
SELECT category, COUNT(product_key) as total_products FROM datawarehouse.gold.dim_products
GROUP BY category
ORDER BY total_products DESC;

category,total_products
Components,127
Bikes,97
Clothing,35
Accessories,29
null,7


In [0]:
%sql
SELECT category, AVG(cost) as avg_cost FROM datawarehouse.gold.dim_products
GROUP BY category
ORDER BY avg_cost DESC;

category,avg_cost
Bikes,949.4432989690722
Components,264.7165354330709
null,28.571428571428573
Clothing,24.8
Accessories,13.172413793103448


In [0]:
%sql
SELECT p.category, SUM(s.sales_amount) as total_rev FROM datawarehouse.gold.dim_products p
RIGHT JOIN datawarehouse.gold.fact_sales s ON p.product_key = s.product_key
GROUP BY p.category
ORDER BY total_rev DESC;

category,total_rev
Bikes,28316272
Accessories,700262
Clothing,339716


In [0]:
%sql
SELECT c.cusotmer_key, c.firstname, c.lastname, SUM(s.sales_amount) as total_rev FROM datawarehouse.gold.dim_customers c
RIGHT JOIN datawarehouse.gold.fact_sales s ON c.cusotmer_key = s.cusotmer_key
GROUP BY c.cusotmer_key, c.firstname, c.lastname
ORDER BY total_rev DESC
LIMIT 25;

cusotmer_key,firstname,lastname,total_rev
1133,Kaitlyn,Henderson,13294
1302,Nichole,Nara,13294
1309,Margaret,He,13268
1132,Randall,Dominguez,13265
1301,Adriana,Gonzalez,13242
1322,Rosa,Hu,13215
1125,Brandi,Gill,13195
1308,Brad,She,13172
1297,Francisco,Sara,13164
434,Maurice,Shan,12914


In [0]:
%sql
SELECT
SUM(p.quantity) as items_sold,
c.country
FROM datawarehouse.gold.dim_customers c
JOIN datawarehouse.gold.fact_sales p ON p.cusotmer_key = c.cusotmer_key
GROUP BY country
ORDER BY items_sold DESC;

items_sold,country
20481,United States
13346,Australia
7630,Canada
6910,United Kingdom
5626,Germany
5559,France
871,n/a


In [0]:
%sql
SELECT
DISTINCT prd.category,
c.country,
SUM(p.quantity) as items_sold
FROM datawarehouse.gold.dim_customers c
JOIN datawarehouse.gold.fact_sales p ON p.cusotmer_key = c.cusotmer_key
INNER JOIN datawarehouse.gold.dim_products prd ON prd.product_key = p.product_key
GROUP BY country, prd.category
ORDER BY country;

category,country,items_sold
Accessories,Australia,7005
Clothing,Australia,1869
Bikes,Australia,4472
Clothing,Canada,1332
Bikes,Canada,924
Accessories,Canada,5374
Accessories,France,3345
Clothing,France,770
Bikes,France,1444
Clothing,Germany,753


In [0]:
%sql
SELECT * FROM datawarehouse.gold.dim_products LIMIT 10;

product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance,cost,product_line,start_date
1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01
3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
6,351,BK-M82B-48,Mountain-100 Black- 48,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01
7,344,BK-M82S-38,Mountain-100 Silver- 38,BI_MB,Bikes,Mountain Bikes,Yes,1912,Mountain,2011-07-01
8,345,BK-M82S-42,Mountain-100 Silver- 42,BI_MB,Bikes,Mountain Bikes,Yes,1912,Mountain,2011-07-01
9,346,BK-M82S-44,Mountain-100 Silver- 44,BI_MB,Bikes,Mountain Bikes,Yes,1912,Mountain,2011-07-01
10,347,BK-M82S-48,Mountain-100 Silver- 48,BI_MB,Bikes,Mountain Bikes,Yes,1912,Mountain,2011-07-01


In [0]:
%sql
SELECT * FROM datawarehouse.gold.dim_customers LIMIT 10;

cusotmer_key,customer_id,customer_number,firstname,lastname,country,marital_status,gender,birthdate,create_date
1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
6,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06
7,11006,AW00011006,Janet,Alvarez,Australia,Single,Female,1976-12-02,2025-10-06
8,11007,AW00011007,Marco,Mehta,Australia,Married,Male,1969-11-06,2025-10-06
9,11008,AW00011008,Rob,Verhoff,Australia,Single,Female,1975-07-04,2025-10-06
10,11009,AW00011009,Shannon,Carlson,Australia,Single,Male,1969-09-29,2025-10-06


In [0]:
%sql
SELECT * FROM datawarehouse.gold.fact_sales LIMIT 10;

order_number,product_key,cusotmer_key,order_date,shipping_date,due_date,sales_amount,quantity,price
SO43697,20,10769,2010-12-29,2011-01-05,2011-01-10,3578,1,3578
SO43698,9,17390,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43699,9,14864,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43700,41,3502,2010-12-29,2011-01-05,2011-01-10,699,1,699
SO43701,9,4,2010-12-29,2011-01-05,2011-01-10,3400,1,3400
SO43702,16,16646,2010-12-30,2011-01-06,2011-01-11,3578,1,3578
SO43703,20,5625,2010-12-30,2011-01-06,2011-01-11,3578,1,3578
SO43704,6,6,2010-12-30,2011-01-06,2011-01-11,3375,1,3375
SO43705,7,12,2010-12-30,2011-01-06,2011-01-11,3400,1,3400
SO43706,17,16622,2010-12-31,2011-01-07,2011-01-12,3578,1,3578


In [0]:
%sql
SELECT p.product_name, 
SUM(s.sales_amount) as total_rev 
FROM datawarehouse.gold.dim_products p
RIGHT JOIN datawarehouse.gold.fact_sales s ON p.product_key = s.product_key
GROUP BY p.product_name
ORDER BY total_rev DESC 
LIMIT 5;

product_name,total_rev
Mountain-200 Black- 46,1373454
Mountain-200 Black- 42,1363128
Mountain-200 Silver- 38,1339394
Mountain-200 Silver- 46,1301029
Mountain-200 Black- 38,1294854


In [0]:
%sql
SELECT * FROM (
SELECT p.product_name, 
SUM(s.sales_amount) as total_rev,
ROW_NUMBER() OVER (ORDER BY SUM(s.sales_amount) DESC) rank_products
FROM datawarehouse.gold.dim_products p
RIGHT JOIN datawarehouse.gold.fact_sales s ON p.product_key = s.product_key
GROUP BY p.product_name) t
WHERE rank_products <= 5;

product_name,total_rev,rank_products
Mountain-200 Black- 46,1373454,1
Mountain-200 Black- 42,1363128,2
Mountain-200 Silver- 38,1339394,3
Mountain-200 Silver- 46,1301029,4
Mountain-200 Black- 38,1294854,5


In [0]:
%sql
SELECT p.product_name, 
SUM(s.sales_amount) as total_rev 
FROM datawarehouse.gold.dim_products p
RIGHT JOIN datawarehouse.gold.fact_sales s ON p.product_key = s.product_key
GROUP BY p.product_name
ORDER BY total_rev 
LIMIT 5;

product_name,total_rev
Racing Socks- L,2430
Racing Socks- M,2682
Patch Kit/8 Patches,6382
Bike Wash - Dissolver,7272
Touring Tire Tube,7440
